# Training and evaluation results

This notebook inspects saved experiment summaries, learning histories, evaluation comparisons, predictions, and error reports. Its default cells only read files; they do not train models or run inference.

Saved experiment bundles are discovered under `artifacts/experiments/`. Evaluation reports are discovered under `reports/`. These directories may be empty in a fresh checkout.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
if not (ROOT / "app").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "app").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

EXPERIMENTS_DIR = ROOT / "artifacts" / "experiments"
REPORTS_DIR = ROOT / "reports"

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

summary_paths = sorted(EXPERIMENTS_DIR.glob("*/summary.json")) if EXPERIMENTS_DIR.is_dir() else []
experiment_rows = []
for summary_path in summary_paths:
    run_dir = summary_path.parent
    summary = read_json(summary_path)
    config_path = run_dir / "training_config.json"
    config = read_json(config_path) if config_path.is_file() else {}
    settings = config.get("settings", {})
    test_path = run_dir / "test_metrics.json"
    test = read_json(test_path) if test_path.is_file() else {}
    experiment_rows.append(
        {
            "run": run_dir.name,
            "model": settings.get("model_name", "unknown"),
            "best_epoch": summary.get("best_epoch"),
            "validation_mean_macro_f1": summary.get("best_validation_mean_macro_f1"),
            "test_mean_macro_f1": test.get("mean_macro_f1"),
            "test_joint_accuracy": test.get("joint_accuracy"),
        }
    )
experiments = pd.DataFrame(experiment_rows)
if experiments.empty:
    print(f"No saved experiment summaries found under {EXPERIMENTS_DIR}.")
else:
    display(experiments.sort_values(["model", "run"]))

In [ ]:
history_paths = sorted(EXPERIMENTS_DIR.glob("*/history.json")) if EXPERIMENTS_DIR.is_dir() else []
if not history_paths:
    print("No saved training histories are available.")
for history_path in history_paths:
    history = read_json(history_path)
    if not history:
        continue
    epochs = [row["epoch"] for row in history]
    figure, axis = plt.subplots(figsize=(8, 4))
    for split in ("train", "validation"):
        axis.plot(epochs, [row[split]["mean_macro_f1"] for row in history], marker=".", label=split)
    axis.set(title=f"Mean macro F1: {history_path.parent.name}", xlabel="Epoch", ylabel="Mean macro F1")
    axis.grid(alpha=0.25)
    axis.legend()
    plt.show()

In [ ]:
comparison_paths = sorted(REPORTS_DIR.rglob("comparison.csv")) if REPORTS_DIR.is_dir() else []
if not comparison_paths:
    print(f"No saved evaluation comparisons found under {REPORTS_DIR}.")
for comparison_path in comparison_paths:
    display(Markdown(f"### Evaluation: `{comparison_path.parent.relative_to(ROOT)}`"))
    display(pd.read_csv(comparison_path))

In [ ]:
prediction_paths = sorted(REPORTS_DIR.rglob("predictions.csv")) if REPORTS_DIR.is_dir() else []
if not prediction_paths:
    print("No saved prediction examples are available.")
for prediction_path in prediction_paths:
    predictions = pd.read_csv(prediction_path)
    display(Markdown(f"### Saved predictions: `{prediction_path.parent.relative_to(ROOT)}`"))
    columns = [column for column in ("text", "true_style", "predicted_style", "true_tone", "predicted_tone", "style_confidence", "tone_confidence") if column in predictions.columns]
    display(predictions[columns].head(10))

In [ ]:
error_paths = sorted(REPORTS_DIR.rglob("errors.csv")) if REPORTS_DIR.is_dir() else []
if not error_paths:
    print("No saved per-model error tables are available.")
for error_path in error_paths:
    errors = pd.read_csv(error_path)
    display(Markdown(f"### Error examples: `{error_path.parent.relative_to(ROOT)}`"))
    if errors.empty:
        print("No joint errors were recorded for this model.")
    else:
        columns = [column for column in ("text", "true_style", "predicted_style", "true_tone", "predicted_tone") if column in errors.columns]
        display(errors[columns].head(10))

## Optional training and evaluation

The following commands are examples only; this notebook does not run them. Choose a new experiment directory for every run. Training writes a standalone bundle under `artifacts/experiments/`; evaluation reports go under `reports/evaluations/`. Neither command changes `artifacts/active_model.json` or replaces the model used by the application.

Train an optional TextCNN experiment:

```bash
uv run python -m scripts.train --model textcnn --run-dir artifacts/experiments/notebook_textcnn_01 --epochs 20
```

Evaluate that saved bundle without retraining:

```bash
uv run python -m scripts.evaluate --checkpoints artifacts/experiments/notebook_textcnn_01 --output reports/evaluations/notebook_textcnn_01 --no-require-all
```